In [30]:
import torch 


In [31]:
D_OBS = 1  # input channels (gray scale) 
H_ENC = 32 # hidden dim in encoder 
D_STC = 16 # represenation dim (encoder output channels) 

# Step 1 : ResNet 5 

involves 
- ResNet5 
- TemporalBatchMixin
- ResidualBlock

In [32]:
from architectures import ResNet5

In [33]:
from einops import rearrange 

observations = torch.randn(32, 1, 10, 64, 64)   # Video 
observations = rearrange(observations, "b c t h w -> (b t) c h w")
print(observations.shape)  # notice we have treated each video (with 10 frames) as 10 images ( 32 x 10 ) = 320 images. 

torch.Size([320, 1, 64, 64])


In [34]:
encoder = ResNet5(D_OBS, H_ENC, D_STC)

In [35]:
with torch.no_grad():
    state = encoder._forward(observations)   # [(B,T), C=D_STC=16, H, W]
    print(state.shape)

# after we finished going through encoder we will rearrange images back to videos 
state = rearrange(state, "(b t) c h w -> b c t h w", b = 32)
print(f'Video: {state.shape}')

torch.Size([320, 16, 64, 64])
Video: torch.Size([32, 16, 10, 64, 64])


### Output shape 

In [36]:
print(state.shape)

torch.Size([32, 16, 10, 64, 64])


# unroll_mode `parallel`

for _ in range(nsteps): 

### Step 2 : StateOnlyPredictor

involves 

- StateOnlyPredictor
- SimplePredictor
- ResUNet

In [37]:
predicted_states = state
print(predicted_states.shape)

torch.Size([32, 16, 10, 64, 64])


In [38]:
from architectures import StateOnlyPredictor, ResUNet
H_PRE = 32   # hidden dim in predictor
action_encoded = None 

In [39]:
predictor = StateOnlyPredictor(
    predictor=  ResUNet(2 * D_STC, H_PRE, D_STC), 
    context_length= 2 
)

print(getattr(predictor, "context_length")) 
print(predictor.context_length)

2
2


Output shape 

In [40]:
with torch.no_grad():
    predicted_states = predictor(predicted_states,a=action_encoded)[:,:,:-1]
    print(predicted_states.shape)

torch.Size([32, 16, 8, 64, 64])


### Step 3: Refeed Ground Truth context on the left 

In [41]:


predicted_states = torch.cat(
    (state[:, :, :predictor.context_length],predicted_states),dim=2
)

print(predicted_states.shape)

torch.Size([32, 16, 10, 64, 64])


# Step 3 : Regularizer 

involves 

- VCLoss

# Step 5: Loss 

In [42]:
print(state.shape )
print(predicted_states.shape)

torch.Size([32, 16, 10, 64, 64])
torch.Size([32, 16, 10, 64, 64])
